[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week4/xhour_dimred_demo.ipynb)

# X-Hour 4: Dimensionality Reduction & Interactive Visualization

**PSYC 51.17: Models of Language and Communication**  
**Week 4 - Thursday X-Hour**

---

## Learning Objectives

By the end of this session, you will:
1. Apply PCA, t-SNE, and UMAP to reduce embedding dimensions
2. Understand the trade-offs between different reduction methods
3. Use HDBSCAN to cluster reduced embeddings
4. Generate human-readable topic labels with BERTopic
5. Create interactive visualizations with datamapplot

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q scikit-learn umap-learn hdbscan bertopic datamapplot sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE
import umap
import hdbscan
from bertopic import BERTopic
import datamapplot
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ All imports successful!")

## Part 1: Loading Data

We'll use the **20 Newsgroups dataset** - a classic text classification benchmark with ~18,000 documents across 20 categories. We'll use a subset of 8 categories for manageable computation.

In [ ]:
# Select 8 diverse categories
categories = [
    'sci.space',
    'sci.med',
    'rec.sport.hockey',
    'rec.sport.baseball',
    'talk.politics.misc',
    'talk.religion.misc',
    'comp.graphics',
    'comp.os.ms-windows.misc'
]

print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    shuffle=True,
    random_state=42,
    remove=('headers', 'footers', 'quotes')
)

documents = newsgroups.data
labels = newsgroups.target
label_names = newsgroups.target_names

print(f"\n✓ Loaded {len(documents)} documents across {len(categories)} categories")
print(f"\nCategories:")
for i, name in enumerate(label_names):
    count = sum(labels == i)
    print(f"  {i}: {name} ({count} docs)")

## Part 2: Creating Embeddings

We'll create document embeddings using TF-IDF followed by SVD dimensionality reduction. This creates dense 100-dimensional representations.

In [ ]:
# Create TF-IDF matrix
print("Creating TF-IDF matrix...")
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.5,
    stop_words='english'
)
tfidf_matrix = tfidf.fit_transform(documents)
print(f"  TF-IDF matrix shape: {tfidf_matrix.shape}")

# Reduce to 100D with SVD
print("\nReducing to 100 dimensions with SVD...")
svd = TruncatedSVD(n_components=100, random_state=42)
embeddings = svd.fit_transform(tfidf_matrix)

print(f"  Embeddings shape: {embeddings.shape}")
print(f"  Explained variance: {svd.explained_variance_ratio_.sum():.2%}")

## Part 3: PCA (Principal Component Analysis)

PCA finds the directions of maximum variance in the data. It's:
- **Linear**: Only captures linear relationships
- **Fast**: Efficient computation via eigendecomposition
- **Global**: Preserves global structure, may lose local clusters

In [ ]:
# Apply PCA
print("Applying PCA...")
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(embeddings)

print(f"  Variance explained: {sum(pca.explained_variance_ratio_):.2%}")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
for i, name in enumerate(label_names):
    mask = labels == i
    ax.scatter(pca_2d[mask, 0], pca_2d[mask, 1], 
               label=name.split('.')[-1], alpha=0.6, s=20)

ax.set_title('PCA: 20 Newsgroups Embeddings', fontsize=14, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 💡 Discussion: PCA Results

- How well separated are the clusters?
- Notice how some categories overlap - why might this be?
- What does the variance explained tell us?

## Part 4: t-SNE (t-distributed Stochastic Neighbor Embedding)

t-SNE is a non-linear technique that:
- **Preserves local structure**: Neighbors stay neighbors
- **Creates tight clusters**: Great for visualization
- **Slow**: O(N²) complexity
- **Non-deterministic**: Different runs may look different

In [ ]:
# Apply t-SNE (use subset for speed)
print("Applying t-SNE (may take a minute)...")

# Subsample for faster computation
n_samples = min(2000, len(embeddings))
indices = np.random.choice(len(embeddings), n_samples, replace=False)
embeddings_subset = embeddings[indices]
labels_subset = labels[indices]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    n_iter=1000,
    random_state=42,
    init='pca'
)
tsne_2d = tsne.fit_transform(embeddings_subset)

print(f"  Processed {n_samples} documents")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
for i, name in enumerate(label_names):
    mask = labels_subset == i
    ax.scatter(tsne_2d[mask, 0], tsne_2d[mask, 1], 
               label=name.split('.')[-1], alpha=0.6, s=20)

ax.set_title('t-SNE: 20 Newsgroups Embeddings', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 💡 Discussion: t-SNE Results

- How do the clusters compare to PCA?
- **Warning**: Cluster sizes in t-SNE are arbitrary! A "bigger" cluster doesn't mean more variance.
- **Warning**: Distances between clusters are meaningless!

## Part 5: UMAP (Uniform Manifold Approximation and Projection)

UMAP is the modern standard (2018). It:
- **Preserves local AND global structure**
- **Fast**: O(N log N) complexity
- **Scalable**: Handles millions of points
- **Supports new data**: Can transform new points without refitting

In [ ]:
# Apply UMAP
print("Applying UMAP...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
umap_2d = reducer.fit_transform(embeddings)

print(f"  UMAP embedding shape: {umap_2d.shape}")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
for i, name in enumerate(label_names):
    mask = labels == i
    ax.scatter(umap_2d[mask, 0], umap_2d[mask, 1], 
               label=name.split('.')[-1], alpha=0.6, s=20)

ax.set_title('UMAP: 20 Newsgroups Embeddings', fontsize=14, fontweight='bold')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 💡 Exercise: Compare the Three Methods

1. Which method produces the most visually distinct clusters?
2. Which method best preserves the relationships between related categories (e.g., sci.space vs sci.med)?
3. When would you use each method?

## Part 6: HDBSCAN Clustering

HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) is the standard for clustering embeddings. It:
- **Finds clusters automatically**: No need to specify k
- **Handles noise**: Points that don't belong to any cluster get label -1
- **Works well with UMAP**: UMAP preserves density, which HDBSCAN uses

In [ ]:
# Apply HDBSCAN to UMAP embeddings
print("Applying HDBSCAN clustering...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
cluster_labels = clusterer.fit_predict(umap_2d)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise = sum(cluster_labels == -1)

print(f"\n✓ Found {n_clusters} clusters")
print(f"✓ Noise points: {n_noise} ({n_noise/len(cluster_labels):.1%})")

# Plot clusters
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    umap_2d[:, 0], umap_2d[:, 1],
    c=cluster_labels, cmap='tab20', alpha=0.6, s=20
)
ax.set_title(f'HDBSCAN Clusters (n={n_clusters})', fontsize=14, fontweight='bold')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

## Part 7: BERTopic for Human-Readable Labels

BERTopic generates human-readable topic labels by analyzing the documents in each cluster. It uses c-TF-IDF to find the most representative words.

In [ ]:
# Create BERTopic model with our pre-computed embeddings and clusters
print("Creating BERTopic model...")

# Create a custom UMAP model that returns pre-computed embeddings
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Use BERTopic with our pre-computed HDBSCAN clusters
topic_model = BERTopic(
    hdbscan_model=clusterer,
    vectorizer_model=CountVectorizer(stop_words='english'),
    ctfidf_model=ClassTfidfTransformer(),
    calculate_probabilities=False,
    verbose=False
)

# Fit with our embeddings
topics, _ = topic_model.fit_transform(documents, embeddings=embeddings)

# Get topic info
topic_info = topic_model.get_topic_info()
print("\n✓ BERTopic model created!")
print(f"\nTopic Summary:")
print(topic_info[['Topic', 'Count', 'Name']].head(15))

In [ ]:
# Show top words for each topic
print("\nTop Words per Topic:")
print("=" * 60)
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    if words:
        top_words = ', '.join([w[0] for w in words[:8]])
        print(f"\nTopic {topic_id}: {top_words}")

### 💡 Discussion: Topic Quality

- Are the topics interpretable? Can you guess what each topic is about?
- Do the BERTopic clusters align with the original newsgroup categories?
- How could we improve the topic labels?

## Part 8: Interactive Visualization with datamapplot

datamapplot creates beautiful, interactive visualizations where you can:
- **Hover** to see document text
- **Search** for specific topics or terms
- **Zoom** into regions of interest

In [ ]:
# Create labels for datamapplot
print("Preparing labels for visualization...")

# Map topic IDs to readable names
topic_names = []
for t in topics:
    if t == -1:
        topic_names.append("Noise")
    else:
        name = topic_info.loc[topic_info.Topic == t, 'Name'].values[0]
        # Simplify the name (remove topic ID prefix)
        topic_names.append(name.split('_', 1)[-1] if '_' in name else name)

# Truncate documents for hover text
hover_texts = [doc[:300] + "..." if len(doc) > 300 else doc for doc in documents]

print(f"\n✓ Prepared {len(topic_names)} labels")

In [ ]:
# Create interactive plot
print("Creating interactive visualization...")

plot = datamapplot.create_interactive_plot(
    umap_2d,
    topic_names,
    hover_text=hover_texts,
    title="20 Newsgroups: Interactive Topic Map",
    sub_title="Explore document clusters and their topics",
    enable_search=True,
    noise_label="Noise",
    darkmode=False
)

print("\n✓ Interactive plot ready!")
print("  - Hover over points to see document text")
print("  - Use the search bar to find specific topics")
print("  - Scroll to zoom in/out")

# Display the plot
plot

## Summary

In this workshop, we learned:

| Method | Pros | Cons | Best For |
|--------|------|------|----------|
| **PCA** | Fast, deterministic, interpretable | Linear only, misses clusters | Quick exploration, preprocessing |
| **t-SNE** | Beautiful cluster visualizations | Slow, non-deterministic | Publication-quality figures |
| **UMAP** | Fast, scalable, global structure | Less interpretable | Interactive dashboards, ML pipelines |

### Best Practice Workflow

```
Raw Embeddings (768D)
      ↓
PCA to 50D (noise reduction)
      ↓
UMAP to 2D (visualization)
      ↓
HDBSCAN (clustering)
      ↓
BERTopic (labeling)
      ↓
datamapplot (interactive viz)
```

## Further Exploration

Try these exercises on your own:

1. **Hyperparameter tuning**: Change `n_neighbors` and `min_dist` in UMAP. How do the clusters change?

2. **Different embeddings**: Replace TF-IDF with SentenceTransformer embeddings. Do the clusters improve?

3. **Outlier reduction**: Use `topic_model.reduce_outliers()` to assign noise points to topics.

4. **Compare with ground truth**: How well do BERTopic clusters align with the original newsgroup labels?